# Mission 2 — MLP optimization bottleneck

clean 59D representation과 기존 Training-only fixed call_id GroupSplit을 고정하고, `max_iter=100`과 `max_iter=300`만 비교합니다. 두 모델은 같은 internal train/validation 행과 같은 train-only StandardScaler를 사용합니다. threshold는 0.5이며 Official Validation과 새 feature는 사용하지 않습니다.

두 학습은 아래 실행 셀에서 순서대로 수행됩니다. 실제 competition-data fit과 결과 해석은 사용자가 이 Notebook에서 진행합니다.

## 1. 기존 clean 59D 로더와 비교 설정

Stage 4에서 쓰던 clean-chain 로더와 기존 MLP 설정을 불러옵니다. 두 설정의 유일한 차이는 `max_iter`입니다.

In [1]:
from pathlib import Path
import sys

import pandas as pd
from IPython.display import display

REPO_ROOT = Path.cwd().resolve()
if REPO_ROOT.name == 'notebooks':
    REPO_ROOT = REPO_ROOT.parent
if not (REPO_ROOT / 'src/dcc_data').is_dir():
    raise RuntimeError('Repository root was not found from the current directory.')
sys.path.insert(0, str(REPO_ROOT / 'src'))

from dcc_data import mission2_model_experiments as models
from dcc_data.mission2_clean_stage4 import load_clean_stage4_59d
from dcc_data.mission2_mlp_bottleneck import (
    build_comparison_delta,
    build_mlp_configs,
    fit_mlp_iteration_candidate,
    prepare_clean_59d,
)

TRAINING_CACHE_PATHS = models.standalone_training_cache_paths(REPO_ROOT)
CLEAN_CHAIN_ROOT = (
    REPO_ROOT
    / 'data/processed/mission2_stage4_clean/stage4_clean_v1_090422f783f0d911'
)
config_100, config_300 = build_mlp_configs()
print('Clean 59D chain:', CLEAN_CHAIN_ROOT)
print('max_iter=100:', config_100)
print('max_iter=300:', config_300)

Clean 59D chain: C:\Users\YW\Study\02_Projects\DCC_2026\prelim\data\processed\mission2_stage4_clean\stage4_clean_v1_090422f783f0d911
max_iter=100: {'hidden_layer_sizes': (64,), 'activation': 'relu', 'solver': 'adam', 'alpha': 0.0001, 'batch_size': 1024, 'learning_rate': 'constant', 'learning_rate_init': 0.001, 'max_iter': 100, 'shuffle': True, 'random_state': 42, 'tol': 0.0001, 'early_stopping': False, 'n_iter_no_change': 10}
max_iter=300: {'hidden_layer_sizes': (64,), 'activation': 'relu', 'solver': 'adam', 'alpha': 0.0001, 'batch_size': 1024, 'learning_rate': 'constant', 'learning_rate_init': 0.001, 'max_iter': 300, 'shuffle': True, 'random_state': 42, 'tol': 0.0001, 'early_stopping': False, 'n_iter_no_change': 10}


## 2. clean 59D 로드

기존 Stage 4 loader가 Training clean 59D와 입력 계약을 확인합니다. Official Validation 데이터는 모델 입력과 평가에 사용하지 않습니다.

In [2]:
clean_59d = load_clean_stage4_59d(
    CLEAN_CHAIN_ROOT,
    TRAINING_CACHE_PATHS.manifest_parquet,
    TRAINING_CACHE_PATHS.manifest_metadata,
    REPO_ROOT / 'data/raw_original',
)
print('Loaded clean Training representation:', clean_59d.X.shape)
print('Feature count:', len(clean_59d.feature_names))

Loaded clean Training representation: (889358, 59)
Feature count: 59


## 3. 기존 fixed split과 train-only scaling

기존 seed 42 GroupSplit helper가 고정 행을 나누고, 기존 representation helper가 internal train에만 scaler를 fit합니다.

In [3]:
prepared = prepare_clean_59d(clean_59d)
print('Fixed call_id split:', dict(prepared.split_audit))
print('Internal train rows used to fit scaler:', prepared.scaler.n_samples_seen_)
print('Scaled matrices:', prepared.X_train_scaled.shape, prepared.X_evaluation_scaled.shape)

Fixed call_id split: {'seed': 42, 'test_size': 0.2, 'total_rows': 889358, 'train_rows': 711467, 'validation_rows': 177891, 'train_calls': 23360, 'validation_calls': 5840, 'call_overlap': 0}
Internal train rows used to fit scaler: 711467.0
Scaled matrices: (711467, 59) (177891, 59)


## 4. 같은 데이터에서 max_iter=100, 300 학습

이 셀은 두 모델을 순서대로 fit합니다. 실제 competition-data 학습은 이 Notebook에서 사용자가 실행합니다.

In [4]:
result_100 = fit_mlp_iteration_candidate(prepared, config_100)
print('Finished max_iter=100')
result_300 = fit_mlp_iteration_candidate(prepared, config_300)
print('Finished max_iter=300')

Finished max_iter=100
Finished max_iter=300


## 5. 두 결과와 핵심 진단 비교

gap은 `train accuracy - internal validation accuracy`, 모든 delta는 `300회 후보 - 100회 기준`입니다. loss 표에는 각 학습의 마지막 10개 iteration까지만 표시합니다.

In [5]:
result_rows = [result_100, result_300]
result_columns = [
    'max_iter_setting',
    'n_iter_',
    'convergence_warning_raised',
    'final_loss',
    'train_accuracy',
    'internal_validation_accuracy',
    'train_validation_gap',
    'confusion_matrix',
    'speaker_0_recall',
    'speaker_1_recall',
    'validation_error_count',
    'fit_time_seconds',
]
display(pd.DataFrame(result_rows).loc[:, result_columns])

delta = build_comparison_delta(result_100, result_300)
display(pd.DataFrame([delta]))

loss_curve_tail = pd.DataFrame(
    [
        {'max_iter_setting': result['max_iter_setting'], 'iteration': iteration, 'loss': loss}
        for result in result_rows
        for iteration, loss in result['loss_curve_tail']
    ]
)
display(loss_curve_tail)
for result in result_rows:
    if result['convergence_warning_raised']:
        print(
            f"max_iter={result['max_iter_setting']} convergence warning:",
            result['convergence_warning_messages'],
        )

,max_iter_setting,n_iter_,convergence_warning_raised,final_loss,train_accuracy,internal_validation_accuracy,train_validation_gap,confusion_matrix,speaker_0_recall,speaker_1_recall,validation_error_count,fit_time_seconds
0,100,100,True,0.266970,0.875788,0.873760,0.002028,"[[72265, 13086], [9371, 83169]]",0.846680,0.898736,22457,299.141957
1,300,130,False,0.265783,0.875892,0.874867,0.001024,"[[70145, 15206], [7054, 85486]]",0.821842,0.923774,22260,423.545996


,validation_accuracy_delta,train_accuracy_delta,gap_delta,error_count_delta
0,0.001107,0.000104,-0.001003,-197


,max_iter_setting,iteration,loss
0,100,91,0.267444
1,100,92,0.267299
2,100,93,0.267241
3,100,94,0.267077
4,100,95,0.267122
5,100,96,0.267122
6,100,97,0.266950
7,100,98,0.266907
8,100,99,0.266743
9,100,100,0.266970


max_iter=100 convergence warning: ("Stochastic Optimizer: Maximum iterations (100) reached and the optimization hasn't converged yet.",)
